# 02 · Data Validation

**Project:** Enterprise HR AI  
**Purpose:** Assert-based internal validation of `employee_attrition.csv` and `Cleaned_HR_Data_Analysis.csv`.  
**Rule:** Files are validated independently. No merge. Asserts raise on violation — no suppression.

---

In [ ]:
import pandas as pd
import numpy as np
import os

RAW = os.path.join('..', 'data', 'raw')
print('Raw path:', os.path.abspath(RAW))
print('Files:', sorted(os.listdir(RAW)))

---
## Section 1 · employee_attrition.csv

In [ ]:
attrition = pd.read_csv(os.path.join(RAW, 'employee_attrition.csv'))
print(f'Loaded employee_attrition.csv: {attrition.shape[0]} rows x {attrition.shape[1]} cols')

### V-ATT-1 · Schema check — 10 required columns exist

In [ ]:
REQUIRED_ATTRITION_COLS = [
    'Age', 'Attrition', 'Department', 'JobRole',
    'EmployeeNumber', 'MonthlyIncome', 'OverTime',
    'JobSatisfaction', 'YearsAtCompany', 'WorkLifeBalance'
]

missing_cols = [c for c in REQUIRED_ATTRITION_COLS if c not in attrition.columns]
if missing_cols:
    print(f'MISSING COLUMNS: {missing_cols}')

assert missing_cols == [], (
    f'V-ATT-1 FAILED — missing columns in employee_attrition.csv: {missing_cols}'
)
print('V-ATT-1 PASSED: all 10 required columns present')

### V-ATT-2 · Type check — Age is integer, MonthlyIncome is numeric

In [ ]:
print(f"Age dtype          : {attrition['Age'].dtype}")
print(f"MonthlyIncome dtype: {attrition['MonthlyIncome'].dtype}")

assert pd.api.types.is_integer_dtype(attrition['Age']), (
    f"V-ATT-2a FAILED — Age is not integer dtype, got: {attrition['Age'].dtype}"
)
print('V-ATT-2a PASSED: Age is integer dtype')

assert pd.api.types.is_numeric_dtype(attrition['MonthlyIncome']), (
    f"V-ATT-2b FAILED — MonthlyIncome is not numeric dtype, got: {attrition['MonthlyIncome'].dtype}"
)
print('V-ATT-2b PASSED: MonthlyIncome is numeric dtype')

### V-ATT-3 · Range check — Age between 18 and 100

In [ ]:
age_min = attrition['Age'].min()
age_max = attrition['Age'].max()
print(f'Age range in employee_attrition: min={age_min}, max={age_max}')

bad_age = attrition[(attrition['Age'] < 18) | (attrition['Age'] > 100)]
if len(bad_age) > 0:
    print(f'Offending rows ({len(bad_age)}):')
    print(bad_age[['EmployeeNumber', 'Age']].to_string())

assert len(bad_age) == 0, (
    f'V-ATT-3 FAILED — {len(bad_age)} rows have Age outside [18, 100]'
)
print('V-ATT-3 PASSED: all Age values within [18, 100]')

### V-ATT-4 · Uniqueness — EmployeeNumber has no duplicates

In [ ]:
dup_empnum = attrition[attrition.duplicated(subset=['EmployeeNumber'], keep=False)]
if len(dup_empnum) > 0:
    print(f'Duplicate EmployeeNumber rows ({len(dup_empnum)}):')
    print(dup_empnum[['EmployeeNumber', 'Age', 'Department']].to_string())

assert len(dup_empnum) == 0, (
    f'V-ATT-4 FAILED — {len(dup_empnum)} rows have duplicate EmployeeNumber'
)
n_unique = attrition['EmployeeNumber'].nunique()
print(f'V-ATT-4 PASSED: EmployeeNumber is unique ({n_unique} distinct values)')

### V-ATT-5 · Category check — Attrition only contains {'Yes', 'No'}

In [ ]:
ALLOWED_ATTRITION = {'Yes', 'No'}
actual_vals = set(attrition['Attrition'].dropna().unique())
print(f'Attrition unique values: {sorted(actual_vals)}')

unexpected = actual_vals - ALLOWED_ATTRITION
if unexpected:
    bad_rows = attrition[~attrition['Attrition'].isin(ALLOWED_ATTRITION)]
    print(f'Offending rows ({len(bad_rows)}):')
    print(bad_rows[['EmployeeNumber', 'Attrition']].to_string())

assert unexpected == set(), (
    f'V-ATT-5 FAILED — unexpected Attrition values: {unexpected}'
)
print(f'V-ATT-5 PASSED: Attrition contains only {ALLOWED_ATTRITION}')

### employee_attrition.csv — Validation Summary

| Check | ID | Status |
|---|---|---|
| Schema (10 required cols) | V-ATT-1 | Ran above |
| Age integer, MonthlyIncome numeric | V-ATT-2 | Ran above |
| Age in [18, 100] | V-ATT-3 | Ran above |
| EmployeeNumber unique | V-ATT-4 | Ran above |
| Attrition ∈ {Yes, No} | V-ATT-5 | Ran above |

---
## Section 2 · Cleaned_HR_Data_Analysis.csv

In [ ]:
hr = pd.read_csv(os.path.join(RAW, 'Cleaned_HR_Data_Analysis.csv'))
print(f'Loaded Cleaned_HR_Data_Analysis.csv: {hr.shape[0]} rows x {hr.shape[1]} cols')

### V-HR-1 · Schema check — 5 required columns exist

In [ ]:
REQUIRED_HR_COLS = [
    'Employee ID', 'Engagement Score',
    'Satisfaction Score', 'Work-Life Balance Score', 'Age'
]

missing_hr = [c for c in REQUIRED_HR_COLS if c not in hr.columns]
if missing_hr:
    print(f'MISSING COLUMNS: {missing_hr}')

assert missing_hr == [], (
    f'V-HR-1 FAILED — missing columns in Cleaned_HR_Data_Analysis.csv: {missing_hr}'
)
print('V-HR-1 PASSED: all 5 required columns present')

### V-HR-2 · Score range discovery — actual min/max printed before asserting any bounds

In [ ]:
SCORE_COLS = ['Engagement Score', 'Satisfaction Score', 'Work-Life Balance Score']

print('=== ACTUAL SCORE RANGES (before asserting any bounds) ===')
score_ranges = {}
for col in SCORE_COLS:
    lo = hr[col].min()
    hi = hr[col].max()
    score_ranges[col] = (lo, hi)
    print(f'  {col:<30s}  min={lo}  max={hi}')

### V-HR-2 (cont.) · Assert — all score values within the observed scale

In [ ]:
# Scale bounds derived from discovery cell above — not hardcoded.
SCALE_LO = int(min(lo for lo, hi in score_ranges.values()))
SCALE_HI = int(max(hi for lo, hi in score_ranges.values()))
print(f'Asserting score bounds: [{SCALE_LO}, {SCALE_HI}] (derived from observed data)')

for col in SCORE_COLS:
    bad = hr[(hr[col] < SCALE_LO) | (hr[col] > SCALE_HI)]
    if len(bad) > 0:
        n_bad = len(bad)
        print(f'Offending rows for [{col}] ({n_bad} rows):')
        print(bad[['Employee ID', col]].to_string())
    assert len(bad) == 0, (
        f'V-HR-2 FAILED — {len(bad)} rows in [{col}] outside [{SCALE_LO}, {SCALE_HI}]'
    )
    print(f'V-HR-2 PASSED: [{col}] all values within [{SCALE_LO}, {SCALE_HI}]')

### V-HR-3 · Uniqueness — Employee ID has no duplicates

In [ ]:
dup_empid = hr[hr.duplicated(subset=['Employee ID'], keep=False)]
if len(dup_empid) > 0:
    print(f'Duplicate Employee ID rows ({len(dup_empid)}):')
    print(dup_empid[['Employee ID', 'Age', 'Performance Score']].to_string())

assert len(dup_empid) == 0, (
    f'V-HR-3 FAILED — {len(dup_empid)} rows have duplicate Employee ID'
)
n_uid = hr['Employee ID'].nunique()
print(f'V-HR-3 PASSED: Employee ID is unique ({n_uid} distinct values)')

### V-HR-4 · Range check — Age between 18 and 100

In [ ]:
hr_age_min = hr['Age'].min()
hr_age_max = hr['Age'].max()
print(f'Age range in Cleaned_HR_Data_Analysis: min={hr_age_min}, max={hr_age_max}')

bad_hr_age = hr[(hr['Age'] < 18) | (hr['Age'] > 100)]
if len(bad_hr_age) > 0:
    print(f'Offending rows ({len(bad_hr_age)}):')
    print(bad_hr_age[['Employee ID', 'Age']].to_string())

assert len(bad_hr_age) == 0, (
    f'V-HR-4 FAILED — {len(bad_hr_age)} rows have Age outside [18, 100]'
)
print('V-HR-4 PASSED: all Age values within [18, 100]')

### Cleaned_HR_Data_Analysis.csv — Validation Summary

| Check | ID | Status |
|---|---|---|
| Schema (5 required cols) | V-HR-1 | Ran above |
| Score cols in observed scale | V-HR-2 | Ran above |
| Employee ID unique | V-HR-3 | Ran above |
| Age in [18, 100] | V-HR-4 | Ran above |

---
## Important Scoping Note

**Employee ID overlap between these two files is 49.7% (731/1470) — validation here only confirms internal validity of each file, it does NOT confirm join correctness. Row-level join validity is a Day 1 §4 (data_relationships) concern, not a data_validation concern.**

Specifically:
- `employee_attrition.csv` has been validated: `EmployeeNumber` is unique and within expected schema/types/ranges.
- `Cleaned_HR_Data_Analysis.csv` has been validated: `Employee ID` is unique, scores are within the observed [1,5] scale, Age is within [18,100].
- Whether the 731 overlapping IDs represent the same physical employees, or whether the ID namespaces are coincidentally numeric, is a **join integrity** question deferred to the data_relationships notebook.
- No cleaning, merging, or imputation has been performed in this notebook.